#**C.Manipulasi Data Dengan Apache Spark SQL**
##**[GROUP BY, INNER JOIN & UNION ]**

##**1. Hadoop & PySpark initialization di Google Colab**

*PySpark* adalah *interface Python* untuk *Apache Spark*. Penggunaan utama *PySpark* adalah untuk bekerja dengan data dalam Bigdata dan untuk membuat pipeline data.

Walaupun *Apache Spark* mendudukung Big Data, sebagai awal pembelajaran tidak perlu menggunakan data yang besar untuk mendapatkan manfaat dari PySpark. Kita bisa temukan bahwa SparkSQL adalah tools yang bagus untuk melakukan analisis data. Penggunaan library Panda menjadi lambat dengan data yang besar Sumber tentang Apache Spark http://spark.apache.org/docs/latest/api/python/



In [1]:
#Init Hadoop & Spark
#=====Begin

#=====End

In [6]:
# Install PySpark and findspark
!pip install pyspark findspark

import findspark
findspark.init()
from pyspark.sql import SparkSession

# Create a SparkSession
spark = SparkSession.builder.appName("SQLQueries").getOrCreate()
print("SparkSession created successfully!")

SparkSession created successfully!


Sekarang SparkSession sudah diinisialisasi, mari kita muat data `sales_retail_2019.csv` dan buat tampilan sementara untuk menjalankan kueri SQL.

In [4]:
# Load the sales_retail_2019.csv data into a Spark DataFrame
sales_retail_2019_df = spark.read.csv("https://raw.githubusercontent.com/rahmadsa/dataset/main/spark/sales_retail_2019.csv", header=True, inferSchema=True)

# Create a temporary view so we can query it with SQL
sales_retail_2019_df.createOrReplaceTempView("sales_retail_2019")

print("Data 'sales_retail_2019.csv' loaded and temporary view created.")
sales_retail_2019_df.show(5)

NameError: name 'spark' is not defined

Lakukan manipulasi data dari dataset pada tautan berikut:

"https://github.com/rahmadsa/dataset/blob/main/ms_produk.csv"

Dengan perintah - perintah SQL berikut pada platform Apache Spark SQL

#**2. Manipulasi Data Dengan GROUP BY Statment**


Lakukan manipulasi data dari dataset pada tautan berikut:

"https://github.com/rahmadsa/dataset/blob/main/spark/sales_retail_2019.csv"

Dengan perintah - perintah SQL berikut pada platform Apache Spark SQL

##2.1. [Group by Single Column](https://#)
```
SELECT province,
COUNT(DISTINCT order_id) AS total_order,
SUM(item_price) AS total_price
FROM sales_retail_2019
GROUP BY province;
```

In [2]:
```sql
SELECT province,
COUNT(DISTINCT order_id) AS total_order,
SUM(item_price) AS total_price
FROM sales_retail_2019
GROUP BY province;
```

SyntaxError: invalid syntax (772076052.py, line 1)

Berikut adalah contoh bagaimana Anda dapat menjalankan kueri SQL yang ada di sel ini (`SIdHuMhceRlR`) menggunakan `spark.sql()`.

In [5]:
sql_query = """
SELECT province,
COUNT(DISTINCT order_id) AS total_order,
SUM(item_price) AS total_price
FROM sales_retail_2019
GROUP BY province
"""

# Execute the SQL query using spark.sql()
result_df = spark.sql(sql_query)

# Display the results
result_df.show()

NameError: name 'spark' is not defined

```markdown
Untuk menjalankan kueri SQL `GROUP BY` dengan benar, mari kita pastikan SparkSession Anda diinisialisasi dan data `sales_retail_2019` dimuat serta tampilan sementaranya dibuat. Berikut adalah langkah-langkahnya:
```

In [7]:
# Inisialisasi SparkSession (jika belum berjalan atau kernel direset)
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("SQLQueries").getOrCreate()
print("SparkSession created successfully!")

SparkSession created successfully!


In [12]:
import os

# URL file CSV
csv_url = "https://raw.githubusercontent.com/rahmadsa/dataset/main/spark/sales_retail_2019.csv"

# Nama file lokal
local_filename = "sales_retail_2019.csv"
local_filepath = os.path.join(os.getcwd(), local_filename)

print(f"Target local path for CSV: {local_filepath}")

# Unduh file CSV jika belum ada
if not os.path.exists(local_filepath):
    print(f"Downloading file from {csv_url} to {local_filepath}...")
    !wget -q $csv_url -O $local_filepath
    if os.path.exists(local_filepath):
        print(f"File '{local_filename}' downloaded successfully.")
    else:
        print(f"Error: File '{local_filename}' was not downloaded.")
else:
    print(f"File '{local_filename}' already exists locally.")

# Muat data sales_retail_2019.csv dari file lokal ke dalam Spark DataFrame
# Menggunakan prefix 'file://' untuk memastikan Spark membaca dari sistem file lokal
try:
    sales_retail_2019_df = spark.read.csv(f"file://{local_filepath}", header=True, inferSchema=True)

    # Trim whitespace from column names
    sales_retail_2019_df = sales_retail_2019_df.toDF(*[c.strip() for c in sales_retail_2019_df.columns])

    # Buat tampilan sementara agar bisa dikueri dengan SQL
    sales_retail_2019_df.createOrReplaceTempView("sales_retail_2019")

    print("Data 'sales_retail_2019.csv' loaded from local file and temporary view created.")
    sales_retail_2019_df.show(5)
except Exception as e:
    print(f"Error loading data with Spark: {e}")
    print("Please ensure SparkSession is initialized and the file is correctly downloaded.")

Target local path for CSV: /content/sales_retail_2019.csv
File 'sales_retail_2019.csv' already exists locally.
Data 'sales_retail_2019.csv' loaded from local file and temporary view created.
+---------+-------------------+-----------+------------------+--------------------+------------+---------+--------+----------+----+
| order_id|         order_date|customer_id|              city|            province|  product_id|    brand|quantity|item_price| _c9|
+---------+-------------------+-----------+------------------+--------------------+------------+---------+--------+----------+----+
|1612339.0|2019-01-01 00:00:00|    18055.0| Jakarta Selatan  | DKI Jakarta        | P0648      | BRAND_C |     4.0| 1934000.0|NULL|
|1612339.0|2019-01-01 00:00:00|    18055.0| Jakarta Selatan  | DKI Jakarta        | P3826      | BRAND_V |     8.0|  604000.0|NULL|
|1612339.0|2019-01-01 00:00:00|    18055.0| Jakarta Selatan  | DKI Jakarta        | P1508      | BRAND_G |    12.0|  747000.0|NULL|
|1612339.0|2019-0

Sekarang, mari kita muat data untuk tabel `tr_penjualan` dan `ms_produk` untuk persiapan menjalankan kueri `INNER JOIN`.

In [16]:
import os
from pyspark.sql.functions import trim

# URL file CSV
tr_penjualan_url = "https://raw.githubusercontent.com/rahmadsa/dataset/main/spark/tr_penjualan.csv"
ms_produk_url = "https://raw.githubusercontent.com/rahmadsa/dataset/main/spark/ms_produk.csv"

# Nama file lokal
tr_penjualan_filename = "tr_penjualan.csv"
ms_produk_filename = "ms_produk.csv"

tr_penjualan_filepath = os.path.join(os.getcwd(), tr_penjualan_filename)
ms_produk_filepath = os.path.join(os.getcwd(), ms_produk_filename)

# Unduh file CSV tr_penjualan jika belum ada
if not os.path.exists(tr_penjualan_filepath):
    print(f"Downloading file from {tr_penjualan_url} to {tr_penjualan_filepath}...")
    !wget -q $tr_penjualan_url -O $tr_penjualan_filepath
    if os.path.exists(tr_penjualan_filepath):
        print(f"File '{tr_penjualan_filename}' downloaded successfully.")
    else:
        print(f"Error: File '{tr_penjualan_filename}' was not downloaded.")
else:
    print(f"File '{tr_penjualan_filename}' already exists locally.")

# Unduh file CSV ms_produk jika belum ada
if not os.path.exists(ms_produk_filepath):
    print(f"Downloading file from {ms_produk_url} to {ms_produk_filepath}...")
    !wget -q $ms_produk_url -O $ms_produk_filepath
    if os.path.exists(ms_produk_filepath):
        print(f"File '{ms_produk_filename}' downloaded successfully.")
    else:
        print(f"Error: File '{ms_produk_filename}' was not downloaded.")
else:
    print(f"File '{ms_produk_filename}' already exists locally.")

# Muat data tr_penjualan.csv
try:
    tr_penjualan_df = spark.read.csv(f"file://{tr_penjualan_filepath}", header=True, inferSchema=True)
    tr_penjualan_df = tr_penjualan_df.toDF(*[c.strip() for c in tr_penjualan_df.columns])
    # Trim whitespace from the 'kode_produk' column data
    tr_penjualan_df = tr_penjualan_df.withColumn("kode_produk", trim(tr_penjualan_df["kode_produk"]))
    tr_penjualan_df.createOrReplaceTempView("tr_penjualan")
    print("Data 'tr_penjualan.csv' loaded and temporary view created.")
    tr_penjualan_df.show(5)
except Exception as e:
    print(f"Error loading tr_penjualan data: {e}")

# Muat data ms_produk.csv
try:
    ms_produk_df = spark.read.csv(f"file://{ms_produk_filepath}", header=True, inferSchema=True)
    ms_produk_df = ms_produk_df.toDF(*[c.strip() for c in ms_produk_df.columns])
    # Trim whitespace from the 'kode_produk' column data
    ms_produk_df = ms_produk_df.withColumn("kode_produk", trim(ms_produk_df["kode_produk"]))
    ms_produk_df.createOrReplaceTempView("ms_produk")
    print("Data 'ms_produk.csv' loaded and temporary view created.")
    ms_produk_df.show(5)
except Exception as e:
    print(f"Error loading ms_produk data: {e}")

File 'tr_penjualan.csv' already exists locally.
File 'ms_produk.csv' already exists locally.
Data 'tr_penjualan.csv' loaded and temporary view created.
+--------------+--------------+-------+-----------+--------------------+---+--------+
|kode_transaksi|kode_pelanggan|no_urut|kode_produk|         nama_produk|qty|   harga|
+--------------+--------------+-------+-----------+--------------------+---+--------+
|        tr-001|     labcust07|    1.0|    prod-01| Kotak Pensil Lab   |5.0| 62500.0|
|        tr-001|     labcust07|    2.0|    prod-03| Flash disk Lab 3...|1.0|100000.0|
|        tr-001|     labcust07|    3.0|    prod-09| Buku Planner Age...|3.0| 92000.0|
|        tr-001|     labcust07|    4.0|    prod-04| Flashdisk Lab 32 GB|3.0| 40000.0|
|        tr-002|     labcust01|    1.0|    prod-03| Gift Voucher Lab...|2.0|100000.0|
+--------------+--------------+-------+-----------+--------------------+---+--------+
only showing top 5 rows
Data 'ms_produk.csv' loaded and temporary view cre

In [17]:
# Re-execute cell 5dd07224 to apply data cleaning
# This cell was already executed in the previous turn, but we instruct re-execution to reflect changes.
# The content of cell 5dd07224 is already updated in the notebook state.

# Note: This is a placeholder to indicate re-execution of the modified cell.
# The actual re-execution will be handled by the environment based on the 'modify_cells' command.
# For the purpose of generating cells, we can skip adding a duplicate code here,
# as the previous 'modify_cells' implies its re-execution.
# Instead, I will directly generate the cell to run the SQL query that depends on it.

In [15]:
# Execute the SQL query from cell ruFdqmuckADA
sql_query_join = """
SELECT tr_penjualan.kode_transaksi, tr_penjualan.kode_pelanggan, tr_penjualan.kode_produk, ms_produk.nama_produk, ms_produk.harga, tr_penjualan.qty, ms_produk.harga * tr_penjualan.qty AS total
FROM tr_penjualan
INNER JOIN ms_produk
ON tr_penjualan.kode_produk = ms_produk.kode_produk
"""

result_join_df = spark.sql(sql_query_join)
result_join_df.show()

+--------------+--------------+-----------+-----------+-----+---+-----+
|kode_transaksi|kode_pelanggan|kode_produk|nama_produk|harga|qty|total|
+--------------+--------------+-----------+-----------+-----+---+-----+
+--------------+--------------+-----------+-----------+-----+---+-----+



In [18]:
# Re-execute the SQL query from cell ruFdqmuckADA (now 04edf2d6) after cleaning the data
sql_query_join = """
SELECT tr_penjualan.kode_transaksi, tr_penjualan.kode_pelanggan, tr_penjualan.kode_produk, ms_produk.nama_produk, ms_produk.harga, tr_penjualan.qty, ms_produk.harga * tr_penjualan.qty AS total
FROM tr_penjualan
INNER JOIN ms_produk
ON tr_penjualan.kode_produk = ms_produk.kode_produk
"""

result_join_df = spark.sql(sql_query_join)
result_join_df.show()

+--------------+--------------+-----------+--------------------+--------+---+---------+
|kode_transaksi|kode_pelanggan|kode_produk|         nama_produk|   harga|qty|    total|
+--------------+--------------+-----------+--------------------+--------+---+---------+
|        tr-001|     labcust07|    prod-01| Kotak Pensil Lab...| 62500.0|5.0| 312500.0|
|        tr-001|     labcust07|    prod-03| Gift Voucher Lab...|100000.0|1.0| 100000.0|
|        tr-001|     labcust07|    prod-09| Buku Planner Age...| 92000.0|3.0| 276000.0|
|        tr-001|     labcust07|    prod-04| Flashdisk Lab 32...| 40000.0|3.0| 120000.0|
|        tr-002|     labcust01|    prod-03| Gift Voucher Lab...|100000.0|2.0| 200000.0|
|        tr-002|     labcust01|    prod-10| Sticky Notes Lab...| 55000.0|4.0| 220000.0|
|        tr-002|     labcust01|    prod-07| Tas Travel Organ...| 48000.0|1.0|  48000.0|
|        tr-003|     labcust03|    prod-02| Flashdisk Lab 64...| 55000.0|2.0| 110000.0|
|        tr-004|     labcust03| 

In [13]:
sql_query_multiple_groupby = """
SELECT province,brand,
COUNT(DISTINCT order_id) AS total_order,
SUM(item_price) AS total_price FROM sales_retail_2019
GROUP BY province, brand;
"""

# Execute the SQL query using spark.sql()
result_multiple_groupby_df = spark.sql(sql_query_multiple_groupby)

# Display the results
result_multiple_groupby_df.show()

+--------------------+---------+-----------+-----------+
|            province|    brand|total_order|total_price|
+--------------------+---------+-----------+-----------+
| unknown            | BRAND_C |          2|  4281000.0|
| Sumatra Selatan    | BRAND_P |          2|   1.0961E7|
| Yogyakarta         | BRAND_Z |          3|  6705000.0|
| Jawa Barat         | BRAND_L |         27|   4.6872E7|
| Bali               | BRAND_M |          3|  6682000.0|
| Sumatra Utara      | BRAND_A |          1|  2568000.0|
| DKI Jakarta        | BRAND_O |         20|   3.6307E7|
| DKI Jakarta        | BRAND_V |         47|   9.8928E7|
| Kalimantan Tengah  | BRAND_D |          2|   1.3975E7|
| Jawa Tengah        | BRAND_J |          5|  5960000.0|
| Jawa Tengah        | BRAND_S |         23|    5.594E7|
| Yogyakarta         | BRAND_N |          7|   1.3972E7|
| Kalimantan Selatan | BRAND_G |          1|  1325000.0|
| Jawa Barat         | BRAND_E |          9|   1.7615E7|
| Sumatra Utara      | BRAND_S 

```markdown
Sekarang, mari kita tulis dan jalankan kueri SQL `GROUP BY` seperti yang diminta:
```

In [9]:
sql_query_groupby = """
SELECT province,
COUNT(DISTINCT order_id) AS total_order,
SUM(item_price) AS total_price
FROM sales_retail_2019
GROUP BY province;
"""

# Eksekusi kueri SQL menggunakan spark.sql()
result_groupby_df = spark.sql(sql_query_groupby)

# Tampilkan hasilnya
result_groupby_df.show()

{"ts": "2026-07-05 09:19:17.689", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[TABLE_OR_VIEW_NOT_FOUND] The table or view `sales_retail_2019` cannot be found. Verify the spelling and correctness of the schema and catalog.\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01", "context": {"errorClass": "TABLE_OR_VIEW_NOT_FOUND"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o27.sql.\n: org.apache.spark.sql.AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `sales_retail_2019` cannot be found. Verify the spelling and correctness of the schema and catalog.\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop use DROP VIEW IF EXI

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `sales_retail_2019` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01; line 5 pos 5;
'Aggregate ['province], ['province, 'COUNT(distinct 'order_id) AS total_order#0, 'SUM('item_price) AS total_price#1]
+- 'UnresolvedRelation [sales_retail_2019], [], false


## 2.2. [Group by Multiple Column](https://#)
```
SELECT province,brand,
COUNT(DISTINCT order_id) AS total_order,
SUM(item_price) AS total_price FROM sales_retail_2019
GROUP BY province, brand;
```

In [11]:
sql_query_multiple_groupby = """
SELECT province,brand,
COUNT(DISTINCT order_id) AS total_order,
SUM(item_price) AS total_price FROM sales_retail_2019
GROUP BY province, brand;
"""

# Execute the SQL query using spark.sql()
result_multiple_groupby_df = spark.sql(sql_query_multiple_groupby)

# Display the results
result_multiple_groupby_df.show()

{"ts": "2026-07-05 09:43:02.122", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `province` cannot be resolved. Did you mean one of the following? [`_c9`, `order_id `, ` brand   `, ` product_id `, ` quantity `]. SQLSTATE: 42703", "context": {"errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o27.sql.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `province` cannot be resolved. Did you mean one of the following? [`_c9`, `order_id `, ` brand   `, ` product_id `, ` quantity `]. SQLSTATE: 42703; line 2 pos 7;\n'Aggregate ['province, 'brand], ['province, 'brand, 'COUNT(distinct 'order_id) AS total_order#71, 'SUM('item_price) AS total_price#72]\n+- SubqueryAlias sales_retail_2019\n   +- View (`sales_retail_2019`, [order_id #19,  

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `province` cannot be resolved. Did you mean one of the following? [`_c9`, `order_id `, ` brand   `, ` product_id `, ` quantity `]. SQLSTATE: 42703; line 2 pos 7;
'Aggregate ['province, 'brand], ['province, 'brand, 'COUNT(distinct 'order_id) AS total_order#71, 'SUM('item_price) AS total_price#72]
+- SubqueryAlias sales_retail_2019
   +- View (`sales_retail_2019`, [order_id #19,  order_date          #20,  customer_id #21,  city             #22,  province           #23,  product_id #24,  brand   #25,  quantity #26,  item_price #27, _c9#28])
      +- Relation [order_id #19, order_date          #20, customer_id #21, city             #22, province           #23, product_id #24, brand   #25, quantity #26, item_price #27,_c9#28] csv


In [ ]:
SELECT;

## 2.3. [Fungsi Aggregate dengan Grouping](https://#)
```
SELECT province,
COUNT(DISTINCT order_id) AS total_unique_order,
SUM(item_price) AS revenue
FROM sales_retail_2019
GROUP BY province;
```

In [ ]:
SELECT ;

#**3. Manipulasi Data Dengan INNER JOIN**

Fungsi dalam SQL adalah blok kode terstruktur yang melakukan tugas tertentu dan dapat dipanggil berkali-kali. Fungsi ini membantu menyederhanakan dan mengoptimalkan kueri SQL, serta memungkinkan penggunaan kembali kode

Lakukan manipulasi data dari dataset pada tautan berikut:

 * ms_item_kategori.csv
 * ms_item_warna.csv

pada repository:

"https://github.com/rahmadsa/dataset/blob/main/spark/"

Dengan perintah - perintah SQL berikut pada platform Apache Spark SQL



##3.1. [SELECT Beberapa Tabel Tanpa JOIN Statment](https://#)
```
SELECT * FROM ms_item_kategori;
SELECT * FROM ms_item_warna ;
```

In [ ]:
SELECT ;

## 3.2. [Menggabungkan Tabel dengan Key Columns](https://#)
```
SELECT * FROM ms_item_kategori, ms_item_warna
WHERE nama_barang = nama_item;
```

In [ ]:
SELECT ;

##3.3 [Bagaimana jika urutan Tabel diubah?](https://#)
```
SELECT * FROM ms_item_warna, ms_item_kategori
WHERE nama_barang=nama_item;
```

In [ ]:
SELECT ;

##3.4. [Menggunakan Prefix Nama Tabel](https://#)
```
  SELECT ms_item_kategori.*, ms_item_warna.*
  FROM ms_item_warna, ms_item_kategori
  WHERE nama_barang=nama_item;
```

In [ ]:
SELECT;

##3.5. [Penggabungan Tanpa Kondisi](https://#)
```
SELECT * FROM ms_item_kategori, ms_item_warna;
```

In [ ]:
SELECT * FROM ms_item_kategori, ms_item_warna;

## 3.6. [tabel tr_penjualan dan tabel ms_produk](https://#)
```
SELECT * FROM tr_penjualan;
SELECT * FROM ms_produk;
```

In [ ]:
SELECT ;

#**4. Manipulasi Data Dengan UNION**

Fungsi dalam SQL adalah blok kode terstruktur yang melakukan tugas tertentu dan dapat dipanggil berkali-kali. Fungsi ini membantu menyederhanakan dan mengoptimalkan kueri SQL, serta memungkinkan penggunaan kembali kode

Lakukan manipulasi data dari dataset pada tautan berikut:

 * tabel_a.csv
 * tabel_b.csv

pada repository:

"https://github.com/rahmadsa/dataset/blob/main/spark/"

Dengan perintah - perintah SQL berikut pada platform Apache Spark SQL



##4.1. [Tabel yang Akan Digabungkan](https://#)
```
SELECT * FROM tabel_a;
SELECT * FROM tabel_b;
```

In [ ]:
SELECT ;

## 4.2. [Menggunakan UNION](https://#)
```
SELECT * FROM tabel_A
UNION
SELECT * FROM tabel_B;
```

In [ ]:
SELECT ;

##4.3 [Menggunakan UNION dengan Klausa WHERE](https://#)
```
SELECT * FROM tabel_A
WHERE kode_pelanggan='labcust03'
UNION
SELECT * FROM tabel_B
WHERE kode_pelanggan='labcust03';
```

In [ ]:
SELECT ;

##4.4. [Menggunakan UNION dan Menyelaraskan Kolom-Kolomnya](https://#)

Lakukan manipulasi data dari dataset pada tautan berikut:

 * Customers.csv
 * Suppliers.csv

pada repository:

"https://github.com/rahmadsa/dataset/blob/main/spark/"

Dengan perintah - perintah SQL berikut pada platform Apache Spark SQL



```
SELECT CustomerName, ContactName, City, PostalCode
FROM Customers
UNION
SELECT SupplierName, ContactName, City, PostalCode
FROM Suppliers;
```

In [ ]:
SELECT ;

#**5. Tugas**

Fungsi dalam SQL adalah blok kode terstruktur yang melakukan tugas tertentu dan dapat dipanggil berkali-kali.

Lakukan manipulasi data dari dataset pada tautan berikut:

* Customers.csv
* Suppliers.csv
* sales_retail_2019.cvs
* tr_penjualan.cvs
* ms_item_warna.cvs
* ms_item_kategori.cvs
* ms_produk.cvs

pada repository:

"https://github.com/rahmadsa/dataset/blob/main/spark/"

Dengan perintah - perintah SQL berikut pada platform Apache Spark SQL

## 5.1 [Tugas Praktek-3.2](https://#)
```
SELECT MONTH(order_date) AS order_month, SUM(item_price) AS total_price,
CASE
    WHEN SUM(item_price) >= 30000000000 THEN 'Target Achieved'
    WHEN SUM(item_price) <= 25000000000 THEN 'Less Performed'
    ELSE 'Follow Up'
END as remark
FROM sales_retail_2019
GROUP BY MONTH(order_date);
```

In [ ]:
SELECT ;

##5.2. [Proyek Pekerjaan - Analisis Penjualan Part 1](https://#)
-- a. Total jumlah seluruh penjualan (total/revenue).
```
SELECT SUM(total) as total
FROM tr_penjualan;
```
-- b. Total quantity seluruh produk yang terjual.
```
SELECT SUM(qty) as qty
FROM tr_penjualan;
```
-- c. Total quantity dan total revenue untuk setiap kode produk.
```
SELECT kode_produk, SUM(qty) as qty, SUM(total) as total
FROM tr_penjualan
GROUP BY kode_produk;
```

In [ ]:
# a. Total jumlah seluruh penjualan (total/revenue).
# b. Total quantity seluruh produk yang terjual.
# c. Total quantity dan total revenue untuk setiap kode produk.


##5.3 [Proyek Pekerjaan - Analisis Penjualan Part 2](https://#)
-- d. Rata - Rata total belanja per kode pelanggan.
```
SELECT kode_pelanggan, AVG(total) AS avg_total
FROM tr_penjualan
GROUP BY kode_pelanggan;
```
-- e. Selain itu,  jangan lupa untuk menambahkan kolom baru
dengan nama ‘kategori’ yang mengkategorikan total/revenue ke dalam
3 kategori: High: > 300K; Medium: 100K - 300K; Low: <100K.
```
SELECT kode_transaksi,kode_pelanggan,no_urut,kode_produk,nama_produk,qty,total,
CASE
    WHEN total > 300000 THEN 'High'
    WHEN total < 100000 THEN 'Low'
    ELSE 'Medium'
END as kategori
FROM tr_penjualan;
```

In [ ]:
#-- d. Rata - Rata total belanja per kode pelanggan.

#-- e Selain itu,  jangan lupa untuk menambahkan kolom baru


##5.4. [Tugas Praktek: Menggunakan INNER JOIN (1/3)](https://academy.dqlab.id/main/livecode/244/407/2051?pr=0)
```
SELECT * FROM ms_item_warna
INNER JOIN ms_item_kategori
ON ms_item_warna.nama_barang = ms_item_kategori.nama_item;
```

In [ ]:
SELECT ;

## 5.5. [Tugas Praktek: Menggunakan INNER JOIN (2/3)](https://#)
```
SELECT *
FROM tr_penjualan
INNER JOIN ms_produk
ON tr_penjualan.kode_produk = ms_produk.kode_produk;
```

In [ ]:
SELECT ;

##5.6. [Tugas Praktek: Menggunakan INNER JOIN (2/3)](https://#)
```
SELECT *
FROM tr_penjualan
INNER JOIN ms_produk
ON tr_penjualan.kode_produk = ms_produk.kode_produk;
```

In [ ]:
SELECT ;

##5.7 [Tugas Praktek: Menggunakan INNER JOIN (3/3)](https://#)

In [ ]:
SELECT tr_penjualan.kode_transaksi,	tr_penjualan.kode_pelanggan,tr_penjualan.kode_produk,ms_produk.nama_produk,ms_produk.harga,tr_penjualan.qty,ms_produk.harga * tr_penjualan.qty AS total
FROM tr_penjualan
INNER JOIN ms_produk
ON tr_penjualan.kode_produk = ms_produk.kode_produk